# LangGraph Agent 

#### Authenticate with AWS

In [ ]:
# !pip install python-dotenv


In [ ]:
from dotenv import load_dotenv
loaded = load_dotenv(dotenv_path="../.env")
print(loaded)

#### Import metrics for evaluation

In [ ]:
import metrics
from metrics import *


for name in dir(metrics):
    val = getattr(metrics, name)
    if isinstance(val, list):
        print(f"{name}:")
        for m in val:
            print(f"  - {m}")
        print()


#### Choose the set of metrics for evaluation

In [ ]:
metrics = all_metrics
print("You've chosen the following metrics for evaluating the LangGraph agent:")
for m in metrics:
    print(f"  - {m}")

In [ ]:
# Import UAEF
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime
print("✓ UAEF imported successfully!")

## Create LangGraph Agent

#### Step 1: Build a simple LangGraph agent

In [ ]:
# !pip install langgraph langchain-aws langchain-core

In [ ]:
# Step 1: Build a simple LangGraph agent
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_aws import ChatBedrock
import boto3
from botocore.config import Config
from uaef.adapters import LangGraphAdapter

# Define a simple tool
@tool
def get_weather(location: str) -> dict:
    """Get the current weather for a location."""
    # Simulate weather data
    return {"temp": 65, "condition": "cloudy", "location": location}

# Define the agent state
class State(TypedDict):
    messages: Annotated[list, add_messages]

# Setup Bedrock LLM
region_name = "us-east-1"
my_config = Config(
    region_name=region_name,
    signature_version='v4',
    retries={'max_attempts': 3, 'mode': 'standard'}
)
bedrock_runtime = boto3.client(service_name="bedrock-runtime", config=my_config)
bedrock_llm = ChatBedrock(
    client=bedrock_runtime,
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    model_kwargs={"max_tokens": 1024, "temperature": 0.0}
)

# Bind tools to LLM
tools = [get_weather]
llm_with_tools = bedrock_llm.bind_tools(tools)

# Define agent node
def agent_node(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Build the graph
graph_builder = StateGraph(State)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges("agent", tools_condition)
graph_builder.add_edge("tools", "agent")
graph = graph_builder.compile()


#### Step 2: Run the LangGraph Agent

In [ ]:

# Step 2: Run the agent
user_input = "What's the weather in Seattle?"
events = []
# for event in graph.stream({"messages": [HumanMessage(content=user_input)]}, stream_mode="values"):
for event in graph.stream({"messages": [HumanMessage(content=user_input)]}, stream_mode="updates"):

    events.append(event)

#### Step 3: Use LangGraph Adapter 

In [ ]:

# Step 3: Transform to UAEF canonical format
langgraph_result = {
    "stream_events": events,
    "session_id": "lg_001",
    "agent_node_name": "agent",
    "tool_node_name": "tools"
}

adapter = LangGraphAdapter()
agent_trace = adapter.transform_to_canonical(langgraph_result)

print(f"✓ Transformed LangGraph output to AgentTrace")
print(f"  Trace ID: {agent_trace.trace_id}")
print(f"  Messages: {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")


# Get final response
final_messages = [e for e in events if "agent" in e]
if final_messages:
    final_msg = final_messages[-1]["agent"]["messages"][-1]
    print(f"\nAgent Response: {final_msg.content}")


In [ ]:
metrics

#### Step 4: Evaluate

In [ ]:
# !pip install ragas datasets deepeval
# !pip install eval_type_backport

In [ ]:
from datetime import datetime
from uaef.api import evaluate
from uaef.models import GroundTruth, ToolCall

# Define ground truth for sample question
ground_truth = GroundTruth(
    expected_output="The weather in Seattle is 65°F and cloudy",
    expected_tool_calls=[
        ToolCall(
            name="get_weather",
            arguments={"location": "Seattle"},
            timestamp=datetime.utcnow()
        )
    ],
    context_documents=["Seattle is a city in Washington state"]
)

# Step 4: Evaluate
result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics
)

# Display evaluation results for LangGraph
print(f"\n{'='*50}")
print("LangGraph - EVALUATION RESULTS")
print(f"Overall Score: {result.overall_score:.2f}")

print(f"\n{'='*50}")
print("METRIC SCORES BY DIMENSION")

for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        print(f"  {metric.metric_name}: {metric.score:.2f}")



## Batch Evaluation
1. Modify `data/ground-truth.xlsx` with your Ground Truth questions and answers.
2. Run batch evaluation by sending queries from Ground Truth to the live LangGraph agent. 

#### Load Ground Truth Q & A

In [ ]:
import pandas as pd
import json
from datetime import datetime, timezone
from uuid import uuid4

from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall, AgentTrace, Message
from uaef.models.message import MessageRole

# Load ground truth data from Excel
excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)

# Convert Excel rows to JSON records for inspection
gt_json = df.to_dict(orient="records")

print(f"✓ Converted {len(gt_json)} rows to JSON")
print(f"✓ Loaded {len(df)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()



#### Reuse LangGraph

In [ ]:
# Reuse the graph and adapter from Step 2 — no agent recreation
adapter = LangGraphAdapter()
traces = []
ground_truths = []

for i, row in enumerate(gt_json):
    query = str(row.get("query", row.get("input", row.get("Question", ""))))
    expected = str(row.get("expected_output", row.get("expected", row.get("Answer", ""))))
    context = str(row.get("context", "")) if pd.notna(row.get("context")) else ""

    # Parse expected tool calls
    expected_tools = []
    raw_tools = row.get("expected_tool_calls", row.get("tools", None))
    if pd.notna(raw_tools) and raw_tools:
        try:
            parsed = json.loads(str(raw_tools)) if isinstance(raw_tools, str) else raw_tools
            if isinstance(parsed, list):
                for t in parsed:
                    expected_tools.append(ToolCall(
                        name=t.get("name", t.get("tool_name", "")),
                        arguments=t.get("arguments", t.get("parameters", {})),
                        timestamp=datetime.now(timezone.utc)
                    ))
        except (json.JSONDecodeError, TypeError):
            pass

    # Send query to the LangGraph agent  
    events = []
    for event in graph.stream({"messages": [HumanMessage(content=query)]}, stream_mode="updates"):
        events.append(event)

    trace = adapter.transform_to_canonical({
        "stream_events": events,
        "session_id": f"batch_lg_{i}",
        "agent_node_name": "agent",
        "tool_node_name": "tools"
    })
    traces.append(trace)
    ground_truths.append(GroundTruth(
        expected_output=expected,
        expected_tool_calls=expected_tools,
        context_documents=[context] if context else []
    ))

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    print(f"  [{i+1}/{len(gt_json)}] {label}")

print(f"\n✓ Ran {len(traces)} queries through the LangGraph agent")

#### Run evaluation on the agent traces

In [ ]:
# Batch evaluate all traces
batch_results = batch_evaluate(
    traces=traces,
    ground_truths=ground_truths,
    metrics=metrics,
    max_workers=4
)

print(f"{'='*50}")
print(f"BATCH RESULTS — LANGGRAPH AGENT")
print(f"{'='*50}")
for i, r in enumerate(batch_results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in batch_results) / len(batch_results)
pr = sum(1 for r in batch_results if r.passed) / len(batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")

#### Export batch evaluation results

In [ ]:
# Save evaluation results
from uaef.utils import save_metric_results

filepath = save_metric_results(batch_results, gt_json, prefix="langgraph_batch_results")
